In [0]:
CREATE OR REPLACE TABLE project.silver.orders_gen_part (
  order_id     INT,
  order_date   DATE,
  amount       DOUBLE,
  country      STRING
  
)
USING DELTA
PARTITIONED BY (year(order_date), month(order_date));


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5650291642937309>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE TABLE project.silver.orders_gen_part (\n  order_id     INT,\n  order_date   DATE,\n  amount       DOUBLE,\n  country      STRING\n  \n)\nUSING DELTA\nPARTITIONED BY (year(order_date), month(order_date));\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False)

In [0]:
CREATE OR REPLACE TABLE project.silver.orders_gen (
  order_id     INT,
  order_date   DATE,
  amount       DOUBLE,
  country      STRING,
  order_year   INT GENERATED ALWAYS AS (year(order_date)),
  order_month  INT GENERATED ALWAYS AS (month(order_date)),
  order_ym STRING GENERATED ALWAYS AS (concat(year(order_date), '-', month(order_date)))
)
USING DELTA
PARTITIONED BY (order_year, order_month);


In [0]:
INSERT INTO project.silver.orders_gen (order_id, order_date, amount, country) VALUES
(1, '2024-01-05', 5000, 'India'),
(2, '2024-01-12', 6000, 'USA'),
(3, '2024-02-10', 5500, 'UK'),
(4, '2024-03-15', 7100, 'Japan'),
(5, '2025-06-20', 9000, 'India');


num_affected_rows,num_inserted_rows
5,5


In [0]:
select * from project.silver.orders_gen

order_id,order_date,amount,country,order_year,order_month,order_ym
1,2024-01-05,5000.0,India,2024,1,2024-1
2,2024-01-12,6000.0,USA,2024,1,2024-1
4,2024-03-15,7100.0,Japan,2024,3,2024-3
5,2025-06-20,9000.0,India,2025,6,2025-6
3,2024-02-10,5500.0,UK,2024,2,2024-2


In [0]:
CREATE OR REPLACE TABLE project.silver.orders_gen_py (
  order_id     INT,
  order_date   DATE,
  amount       int,
  country      STRING,
  order_year   INT GENERATED ALWAYS AS (year(order_date)),
  order_month  INT GENERATED ALWAYS AS (month(order_date))
)
USING DELTA
PARTITIONED BY (order_year, order_month);


In [0]:
%python
from pyspark.sql.functions import to_date,col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType,DateType
data = [
    (10, "2024-04-01", 8800, "Germany"),
    (11, "2024-04-18", 5300, "India"),
    (12, "2025-01-05", 4700, "USA")
]



schema = StructType([
    StructField('order_id', IntegerType(), True),
    StructField('order_date', StringType(), True),
    StructField('amount', IntegerType(), True),
    StructField('country', StringType(), True)
])
df = spark.createDataFrame(data, schema)
df = df.withColumn('order_date', to_date('order_date', 'yyyy-MM-dd'))
df = df.withColumn('amount', col('amount').cast(IntegerType()))




df.write.format("delta").mode("append").saveAsTable("project.silver.orders_gen_py")


In [0]:
select * from project.silver.orders_gen_py

order_id,order_date,amount,country,order_year,order_month
10,2024-04-01,8800,Germany,2024,4
11,2024-04-18,5300,India,2024,4
12,2025-01-05,4700,USA,2025,1


In [0]:
insert into project.silver.orders_gen_py values(13,'2024-01-05',5000,'india', 2024,1)

num_affected_rows,num_inserted_rows
1,1
